# This notebook is for demonstrating some example use cases of the package

### The examples below also assume that you've already generated some runs using the techniques described in the README.md 

In [ ]:
# All can be directly imported from source, but seperated out here for the sake of transparancy
# Also the relevent imports are put in each cell
import numpy as np
from src.Simulators import BoidSimulator, LorenzSimulator
from src.Visualisers import swarm_render,torus_swarm_render, plot_consistency_profile, plot_ridge_predictions
from src.ConfigManager import compare_params,generate_config,load_config
from src.SaverLoader import Saver, create_mmap, load_run, load_npzs, load_prediction, load_cc, load_readout
from src.ObservationAnalysis import ReadoutMethod, KernelReadout, NaiveReadout, COMReadout, NoReadout

# Generating Runs

##### I'd advise that you attempt to generate runs from the command line because thats the intended method so you're less likely to run into bugs that way. This process is outlined in `README.md`

In [ ]:
from src.ConfigManager import generate_config

# Run this cell if you don't have a config.ini file yet! Warning, it may overwrite similarily named .ini files
# Bear in mind also that you can name your .ini file whatever you like, as long as the internals follow the same formatting


# Write a config file to the testing directory 
testing_dir = './tests/'
generate_config(testing_dir)

In [ ]:
from src.SaverLoader import Saver
# Run this cell to initialise a Saver class instance.
# This class provides methods for saving runs, predictions and readouts
# In this case, initialising it with testing_dir allows it to automatically create organisational folders inside the directory
# It will also be used later for saving runs etc
saver = Saver(testing_dir)

In [ ]:
from src.Simulators import BoidSimulator, LorenzSimulator
from src.ConfigManager import load_config

testing_dir = './tests'
path_to_ini = './tests/config.ini'

# Load your config file using functions provided by ConfigManager.ini
simulation_config = load_config(path_to_ini)

# Generate a driving signal, in this case, the Lorenz System whose generation is included in the package
lorenz_system = LorenzSimulator(simulation_config['l_sigma'],simulation_config['l_rho'],simulation_config['l_beta'])
l_xy = lorenz_system.generate_lorenz(simulation_config['simulation_steps'], simulation_config['l_sampling_rate'], simulation_config['x_lorenz'], simulation_config['y_lorenz'], simulation_config['z_lorenz'])

# Initialise the simulater, passing the config loaded from the .ini
boid_simulator = BoidSimulator(config=simulation_config,driving_signal=l_xy,use_random_seed=True)

In [ ]:
# Generate 2 simulations
for _ in range(2):
    # Run a simulation
    result = boid_simulator.run_simulation()

    # Save the simulation when its complete
    # Optionally, the prefix of the saved file can be set to help keep things organised. Otherwise it'll name itself accordingly
        # Note: Don't worry about ensuring unique names yourself! Saver handles this automatically.
    # The saver instance should automatically a run to its corresponding directory, but if theres an issue, it will fallback to the directory it was initialised with
    saver.save_run(result,"awesome")

In [ ]:
# In the case where a simulation is very long (bigger than can exist all at once in memory)
# use memory mapping!

# Since our simulator was initialised to not use memory mapping, modifying the attribute
boid_simulator.memory_mapping = True
boid_simulator.chunk_size = 100 # using 100 because default config only generates a run of size 500, though chunks size is only limited by memory.

# Run simulation (memmaping now enabled)
run_memmap = boid_simulator.run_simulation()

# Memory mapping is used on the position and velocit values
print("See I'm a np.memmap! -> ",type(run_memmap['positions']))
# Temporary file location, although this will have already be outputed when the temporary files were created
print(run_memmap['velocities'].filename)

In [ ]:
# Save your memmap run as normal, optionally delete the associated temporary npy (default is true)
saver.save_run(run_memmap,prefix="memmap",cleanup_tmps=True)

# Performing Reservoir Readouts (loading runs)

In [ ]:
from src.SaverLoader import load_run
# For use in being analysed, we can load a run we generated earlier
run = load_run('./tests/runs/awesome_run.npz')
print(run.keys())

In [ ]:
from src.SaverLoader import load_npzs,load_run

# Instead of loading every run individually, we can load all of them
runs_path = './tests/runs/'

#we need to define what kind of load function to use i.e. what kind of npz we're loading

run_datas = load_npzs(runs_path,load_function=load_run)

#If there are multiple kinds of npz (runs, readouts, ccs etc) in the same directory, load_npzs will likely fail
#Unfortunatly, automatic detection of npz type fell outside of the scope off this project. Since I'd already generated my data, tagging NPZs would break backwards compatability

In [ ]:
from src.ObservationAnalysis import ReadoutMethod, KernelReadout, NaiveReadout, COMReadout, NoReadout
# The package includes a set of predifined readout methods. It is of course encouraged to define your own like the example below

class CrazyReadout(ReadoutMethod):
    def __init__(self,replica1,replica2,washout,chunk_size):
        super().__init__(replica1,replica2,washout,chunk_size)
    
    
    # The only additional requirement is that the class defines a _create_readout method
    # This is used to create readouts from the class replica attributes
    def _create_readout(self,replica):
        # It may assume that the replica is a certain data structure
        x = replica['positions']
        simulation_steps,num_boids,features = x.shape

        #Randomly adjust every position
        positions_randomised = x + np.random.uniform(-100, 100, size=(simulation_steps, num_boids, features))
        # Multiply each x and y coordinate, (in this case serves the purpose of flattening the readout)
        readout = np.array(x[:,:,0]*x[:,:,1])
        
        # Remember that a readout must be a vector for each time step of the simulation, otherwise it cant be used for ridge regression.
        
        return readout # Im SURE this readout method will be a fantastic for ridge prediction !!!

cr = CrazyReadout(run_datas[0],run_datas[1],washout=0,chunk_size=10) # Initialise our a class instance of our custom readout
readouts = cr.get_reservoir_readout() # Readout the replicas
print(readouts[0].keys()) # Evidence it worked! 
print(readouts[0]['readout'].shape)

# Uncomment below to see the actual readout array created
#print(readouts[0]['readout'])


## Naive Readout + Ridge Regression Example
This readout method is given by concatenating the x and y coordinates at each time step into a single vector giving $(T,2N)$, where $T$ is the number of simulation steps, and $N$ is the number of boids

In [ ]:
runs_path = './tests/runs/'
run_datas = load_npzs(runs_path,load_function=load_run)

# Initialising a NaiveReadout class instance, giving two of the loaded runs as replicas
# The washout defines a period of time to be removed from the beginning of the replicas, in the case of the Lorenz System, this accounts for the transient period of the trajectory before it falls into the attractor
# Chunk size is used when the runs were loaded with memory mapping (see later examples) but in this case isn't relevent
nr = NaiveReadout(replica1=run_datas[0],replica2=run_datas[1],washout=100,chunk_size=10)

In [ ]:
# Calculate the readouts for both replicas
readouts = nr.get_reservoir_readout()

# we can optionally pass one of the class instances replicas to only calculate the state vector for one which is helpful if the replicas are very large and you want to save compute time
#readouts = nr.get_reservoir_readout(nr.replica1)


In [ ]:
# This cell shows how a replicas linear readout can be used in ridge regression, attempted to be fit against the future states of the driving signal (predator x coordinates)
replica1_readout = readouts[0]['readout']

# We define the number of simulation steps in the future for which the linear readout is attempted to be fit against
prediction_distance = 1 

# These are the alpha values used in the cross validation step, although a single scaler value can be used to skip cross validation 
alpha_search = [0.1,1,10,100,1000]

prediction = nr.ridge_prediction(linear_readout=replica1_readout, prediction_distance=1,ridge_alpha=alpha_search)

# Prediction is a dict and contains the following data:
print(prediction.keys())

# See what alpha value the validation step optimised for
print(prediction['alpha'])

# Save the prediction
saver.save_prediction(prediction)

In [ ]:
# Loading the prediction to show how it can be done even though not technically necessary
prediction_path = runs_path = './tests/predictions/'
prediction = load_npzs(prediction_path,load_function=load_prediction)[0]

In [ ]:
## Possible kwargs in here
kwargs = {
    "dpi": 100,
    "plot_ratio": (2, 1),
    "fig_width": 20,
    "grid":False,
    "font_size": 34,

    "y_ticks": [-5, 0, 5],
    "x_ticks": range(0,1250,250),
    "axes_linewidth":2,

    "delta_t":None,

    "signal_label": r"$\bar{x}_L$",
    "signal_color": "black",
    "signal_linestyle":"solid",
    "signal_linewidth":2.5,


    "predictions_line_styles":"solid",
    "predictions_labels":None,
    "predictions_colors":None,
    "predictions_linewidths":1.5,
}

In [ ]:
from src.Visualisers import plot_ridge_predictions
# When plotting its helpful to have the actual y_test that was created along with the prediction
y_test = prediction['y_test']
prediction_distance = prediction['prediction_distance']

# The reason this taken as a seperate parameter, where things like the prediction distance are gotten from the prediction array, is to allow for the option to easily plot a different y_test

plot_ridge_predictions(prediction,y_test, prediction_distance,**kwargs)

## Centre-of-mass (COM) readout, consistency profile example (Faithful Approach), saving consistency profiles

This readout method is given by averaging the $x$,$y$ coordinates of all the boids at a given time step producing $(T,2)$ where $T$ is the number of simulation steps, and $2$ corresponds to the average x and the average y.

In [ ]:
# The operations below are explained in another the Naive Readout section

runs_path = './tests/runs/'
run_datas = load_npzs(runs_path,load_function=load_run)

cr = COMReadout(replica1=run_datas[0],replica2=run_datas[1],washout=100,chunk_size=10)
com_readouts = cr.get_reservoir_readout()

In [ ]:
replica1_readout = com_readouts[0]['readout']
replica2_readout = com_readouts[1]['readout']

# Calculate the consistency profile from the readouts made from the replica pair
cc = cr.calc_consistency_profile(replica1_readout,replica2_readout,methodology="v1")

# cc is a dict and contains the following data:
print(cc.keys())

# We can see the consistent capacity of the replicas given the com readout
print(cc['consistent_capacity'])


In [ ]:
# We can plot the consistency profile
com_consistency_profile = cc['consistency_profile']
com_consistent_capacity = cc['consistent_capacity']

#truncated to 2 because COM readout only has two modes
plot_consistency_profile(com_consistent_capacity,com_consistency_profile,truncated_to=2)

In [ ]:
# We can also save consistency profile
saver.save_cc(cc)

In [ ]:
# This cell shows how two profiles can be plotted against eachother


# Generate a new naive readout and get its faithful consistency profile
nr = NoReadout(replica1=run_datas[0],replica2=run_datas[1],washout=100,chunk_size=10)
replica_readouts = nr.get_reservoir_readout()
replica_readout0 = replica_readouts[0]['readout']
replica_readout1 = replica_readouts[1]['readout']

cc_naive = nr.calc_consistency_profile(replica_readout0,replica_readout1,methodology="faithful")
naive_consistency_profile = cc_naive['consistency_profile']
naive_consistent_capacity = cc_naive['consistent_capacity']

#Combine the c_profiles and ccs into a list
combined_profiles = [naive_consistency_profile, com_consistency_profile]
combined_capacity = [naive_consistent_capacity, com_consistent_capacity]

#plot them together
plot_consistency_profile(combined_capacity,combined_profiles)

## Kernel Readout, Memory Mapping, saving readouts and CCs

In [ ]:
run_paths = './tests/runs'
#using memory mapping (explained in previous section) because the kernel readouts are usually a lot more memory intensive
datas = load_npzs(run_paths,memory_map=True)

# for the kernel readout, you need to specify the number of observation kernels you wish to generate
kr = KernelReadout(datas[0],datas[1],kernel_number=200,washout=100,chunk_size=100)

In [ ]:
# We can pass replicas individually
kernel_vector_1 = kr.get_reservoir_readout(kr.replica1)[0]
kernel_vector_2 = kr.get_reservoir_readout(kr.replica2)[0]

# we can use the Saver class to also save readouts
saver.save_readout(kernel_vector_1)
saver.save_readout(kernel_vector_2)


In [ ]:
# we can see how well this kernel readout predicts the lorenz
lookahead = 25
prediction_dict = kr.ridge_prediction(kernel_vector_1.get('readout'),prediction_distance=lookahead)

pred = prediction_dict['prediction']
corr_coef = prediction_dict['corr_coef']

kr.plot_ridge_prediction(pred,corr_coef,prediction_distance=lookahead,x_range=[0,1000])